## 1. Objective

This notebook establishes baseline model performance before correcting the class imbalance. Logistic regression, decision tree, and random forest models will be trained using the original imbalanced training data and evaluated on the validation set.

F1 is the primary evaluation metric, while AUC-ROC is used to measure the models' ability to rank churned customers above retained customers across classification thresholds. The test set will remain untouched until final evaluation.

## 2. Load processed train and validation sets

In [ ]:
from pathlib import Path
import pandas as pd

project_root = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)
processed_dir = project_root / 'data' / 'processed'

In [ ]:
X_train = pd.read_csv(processed_dir/ 'X_train.csv')
X_val = pd.read_csv(processed_dir/ 'X_val.csv')

y_train = pd.read_csv(processed_dir/ 'y_train.csv')['Exited']
y_val = pd.read_csv(processed_dir/ 'y_val.csv')['Exited']

## 3. Verify shapes, columns, and target proportions

In [ ]:
# Verify the imported data

print(f'X_train shape: {X_train.shape}')
print(f'X_val shape: {X_val.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_val shape: {y_val.shape}')

In [ ]:
# Check the columns of the imported data

X_train.columns.tolist()

In [ ]:
# Integrity check

assert X_train.columns.equals(X_val.columns)
assert X_train.isna().sum().sum() == 0
assert X_val.isna().sum().sum() == 0
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)

forbidden_columns = {
    'RowNumber',
    'CustomerId',
    'Surname',
    'Geography',
    'Gender'
}

assert forbidden_columns.isdisjoint(X_train.columns)

print('Training and validation data loaded successfully.')

In [ ]:
# Check that the original imbalance remains

print('Training target proportions:')
print(y_train.value_counts(normalize=True).round(4))

print('\nValidation target proportions:')
print(y_val.value_counts(normalize=True).round(4))

## 4. Dummy Classifier Baseline

A dummy classifier provides a minimum performance reference. Using the `most_frequent` strategy, it predicts the majority class for every customer without learning relationships between the features and churn.

Because the target is imbalanced, this model may have relatively high accuracy while producing an F1 score of zero for the churn class.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay
)

In [ ]:
# Train the dummy model using the training data
dummy_model = DummyClassifier(
    strategy='most_frequent',
    random_state=42
)

dummy_model.fit(X_train, y_train)

In [ ]:
# Validation predictions

dummy_predictions = dummy_model.predict(X_val)
dummy_probabilities = dummy_model.predict_proba(X_val)[:, 1]

In [ ]:
# Dummy metrics

dummy_results = {
    'Model': 'Dummy Classifier',
    'Accuracy': accuracy_score(y_val, dummy_predictions),
    'Precision': precision_score(
        y_val,
        dummy_predictions,
        zero_division=0
    ),
    'Recall': recall_score(
        y_val,
        dummy_predictions,
        zero_division=0
    ),
    'F1': f1_score(
        y_val,
        dummy_predictions,
        zero_division=0
    ),
    'AUC-ROC': roc_auc_score(
        y_val,
        dummy_probabilities
    )
}

pd.DataFrame([dummy_results]).round(4)

In [ ]:
# Confusion matrix

import matplotlib.pyplot as plt
ConfusionMatrixDisplay.from_predictions(
    y_val,
    dummy_predictions,
    display_labels=['Retained', 'Churned'],
    cmap='Blues'
)

plt.title('Dummy Classifier Confusion Matrix')
plt.show()

### Dummy-classifier findings

The dummy classifier achieved approximately 79.7% accuracy because it predicted the majority class for every customer. However, it did not identify any churned customers, resulting in precision, recall, and F1 scores of zero.

Its AUC-ROC score of 0.50 indicates no ability to distinguish churned customers from retained customers. This demonstrates why accuracy alone is misleading for this imbalanced classification problem. A useful trained model must achieve an F1 score above zero and an AUC-ROC above 0.50.

## 5. Logistic regression

Logistic regression provides the first trained baseline model. It estimates the probability that a customer will churn based on a linear combination of the processed features.

No class weighting or resampling is applied in this notebook, allowing us to observe how the original class imbalance affects the model.

In [ ]:
# Train the logistic regression model using the training data

from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

logistic_model.fit(X_train, y_train)

In [ ]:
# Validation predictions

logistic_predictions = logistic_model.predict(X_val)
logistic_probabilities = logistic_model.predict_proba(X_val)[:, 1]

In [ ]:
# Logistic regression metrics

logistic_results = {
    'Model': 'Logistic Regression',
    'Accuracy': accuracy_score(
        y_val,
        logistic_predictions
    ),
    'Precision': precision_score(
        y_val,
        logistic_predictions,
        zero_division=0
    ),
    'Recall': recall_score(
        y_val,
        logistic_predictions,
        zero_division=0
    ),
    'F1': f1_score(
        y_val,
        logistic_predictions,
        zero_division=0
    ),
    'AUC-ROC': roc_auc_score(
        y_val,
        logistic_probabilities
    )
}

pd.DataFrame([logistic_results]).round(4)

In [ ]:
# Confusion matrix

ConfusionMatrixDisplay.from_predictions(
    y_val,
    logistic_predictions,
    display_labels=['Retained', 'Churned'],
    cmap='Blues'
)

plt.title('Logistic Regression Confusion Matrix')
plt.show()

In [ ]:
# Compare the results of the dummy model and logistic regression

baseline_results = pd.DataFrame([
    dummy_results,
    logistic_results
])

baseline_results.round(4)

### Logistic-regression findings

Logistic regression performed better than the dummy classifier, achieving an F1 score of approximately 0.316 and an AUC-ROC of approximately 0.756.

Its precision of approximately 0.587 indicates that a majority of customers predicted to churn actually churned. However, its recall was only approximately 0.216: the model identified 88 of the 407 churned customers and missed 319.

The relatively high accuracy of approximately 0.810 is influenced by the majority class. The low recall and F1 score indicate that, at the default classification threshold, the imbalanced model favors predicting that customers will remain. Its AUC-ROC suggests that it still has useful ranking ability, which may benefit from imbalance correction or threshold adjustment in a later notebook.

## 6. Decision tree

A decision tree can learn nonlinear relationships and interactions without assuming that their effects are linear.

This baseline tree is trained without class weighting, resampling, pruning, or hyperparameter tuning. Its unrestricted growth also allows us to examine whether the model overfits the training data.

In [ ]:
# Train the decision tree model using the training data

from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

decision_tree_model.fit(X_train, y_train)

In [ ]:
# Validation predictions

tree_predictions = decision_tree_model.predict(X_val)
tree_probabilities = decision_tree_model.predict_proba(X_val)[:, 1]

In [ ]:
# Decision tree metrics

tree_results = {
    'Model': 'Decision Tree',
    'Accuracy': accuracy_score(
        y_val,
        tree_predictions
    ),
    'Precision': precision_score(
        y_val,
        tree_predictions,
        zero_division=0
    ),
    'Recall': recall_score(
        y_val,
        tree_predictions,
        zero_division=0
    ),
    'F1': f1_score(
        y_val,
        tree_predictions,
        zero_division=0
    ),
    'AUC-ROC': roc_auc_score(
        y_val,
        tree_probabilities
    )
}

pd.DataFrame([tree_results]).round(4)

In [ ]:
# Confusion matrix

ConfusionMatrixDisplay.from_predictions(
    y_val,
    tree_predictions,
    display_labels=['Retained', 'Churned'],
    cmap='Blues'
)

plt.title('Decision Tree Confusion Matrix')
plt.show()

In [ ]:
# Tree complexity

print(f'Tree depth: {decision_tree_model.get_depth()}')
print(f'Number of leaves: {decision_tree_model.get_n_leaves()}')

In [ ]:
# Check for overfitting (compare training and validation F1)

tree_train_predictions = decision_tree_model.predict(X_train)

tree_train_f1 = f1_score(
    y_train,
    tree_train_predictions
)

tree_validation_f1 = f1_score(
    y_val,
    tree_predictions
)

print(f'Training F1: {tree_train_f1:.4f}')
print(f'Validation F1: {tree_validation_f1:.4f}')



In [ ]:
# Comparison table updated

baseline_results = pd.DataFrame([
    dummy_results,
    logistic_results,
    tree_results
])

baseline_results.round(4)

### Decision-tree findings

The unrestricted decision tree achieved a validation F1 score of approximately 0.487, outperforming logistic regression's F1 of approximately 0.316. Its recall increased substantially to approximately 0.486, meaning it identified 198 of the 407 churned validation customers.

However, its precision was approximately 0.488, indicating that roughly half of its churn predictions were incorrect. Its AUC-ROC of approximately 0.678 was also lower than logistic regression's 0.756, suggesting that logistic regression ranked customers by churn risk more effectively across thresholds.

The tree reached a depth of approximately 23 with 870 leaves and achieved perfect training performance, while its validation F1 remained below 0.50. This large training-validation gap indicates substantial overfitting. Tree depth and other complexity parameters will be tuned later; no correction is applied during baseline modeling.

## 7. Random forest

A random forest combines predictions from multiple decision trees trained on different bootstrap samples and feature subsets. This generally reduces the variance and overfitting associated with a single decision tree.

This baseline uses the default hyperparameters and does not apply class weighting, resampling, or threshold adjustment.

In [ ]:
# Train the random forest model using the training data

from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    random_state=42,
)
random_forest_model.fit(X_train, y_train)

In [ ]:
# Validation predictions

forest_predictions = random_forest_model.predict(X_val)
forest_probabilities = random_forest_model.predict_proba(X_val)[:, 1]

In [ ]:
# Metrics for the random forest model

forest_results = {
    'Model': 'Random Forest',
    'Accuracy': accuracy_score(
        y_val,
        forest_predictions
    ),
    'Precision': precision_score(
        y_val,
        forest_predictions,
        zero_division=0
    ),
    'Recall': recall_score(
        y_val,
        forest_predictions,
        zero_division=0
    ),
    'F1': f1_score(
        y_val,
        forest_predictions,
        zero_division=0
    ),
    'AUC-ROC': roc_auc_score(
        y_val,
        forest_probabilities
    )
}

pd.DataFrame([forest_results]).round(4)


In [ ]:
# Confusion matrix

ConfusionMatrixDisplay.from_predictions(
    y_val,
    forest_predictions,
    display_labels=['Retained', 'Churned'],
    cmap='Blues'
)

plt.title('Random Forest Confusion Matrix')
plt.show()

In [ ]:
# Check for overfitting

forest_train_predictions = random_forest_model.predict(X_train)

forest_train_f1 = f1_score(
    y_train,
    forest_train_predictions
)

forest_validation_f1 = f1_score(
    y_val,
    forest_predictions
)

print(f'Training F1: {forest_train_f1:.4f}')
print(f'Validation F1: {forest_validation_f1:.4f}')

In [ ]:
# Comparison table updated

baseline_results = pd.DataFrame([
    dummy_results,
    logistic_results,
    tree_results,
    forest_results
])

baseline_results.sort_values(
    by='F1',
    ascending=False
).round(4)

### Random-forest findings

The random forest produced the strongest baseline performance, achieving a validation F1 score of approximately 0.587 and an AUC-ROC of approximately 0.855.

Its precision of approximately 0.792 was considerably higher than that of the other trained baseline models, meaning most customers it classified as churners actually churned. Its recall was approximately 0.467, so it still missed 217 of the 407 churned customers.

Although its validation F1 is just below the project's required final test score of 0.59, the model has not yet been tuned and the class imbalance has not been addressed. Its perfect training F1 also suggests overfitting. Random forest is therefore the strongest candidate for hyperparameter tuning and imbalance correction in the next notebook.

## 8. Validation evaluation 

The four baseline models are compared using the same validation set. F1 is the primary selection metric because the target is imbalanced, while AUC-ROC measures each model's ability to rank churned customers above retained customers across classification thresholds.

Accuracy is retained for context but is not used as the primary model-selection metric.

In [ ]:
validation_results = (
    pd.DataFrame([
        dummy_results,
        logistic_results,
        tree_results,
        forest_results
    ])
    .sort_values(by='F1', ascending=False)
    .reset_index(drop=True)
)

validation_results.round(4)

In [ ]:
comparison_metrics = [
    'Precision',
    'Recall',
    'F1',
    'AUC-ROC'
]

ax = (
    validation_results
    .set_index('Model')[comparison_metrics]
    .plot(
        kind='barh',
        figsize=(10, 6)
    )
)

ax.set_title('Baseline Model Performance on Validation Data')
ax.set_xlabel('Metric Score')
ax.set_ylabel('Model')
ax.set_xlim(0, 1)
ax.legend(title='Metric', loc='lower right')

plt.tight_layout()
plt.show()

### Validation findings

The random forest produced the strongest overall validation performance. It achieved the highest F1 score (0.587), AUC-ROC (0.855), precision (0.792), and accuracy (0.867) among the trained baseline models.

The decision tree achieved slightly higher recall than the random forest (0.486 versus 0.467), but its lower precision and AUC-ROC resulted in weaker overall performance. Its perfect training F1 and substantially lower validation F1 also indicate considerable overfitting.

Logistic regression achieved an AUC-ROC of 0.756, indicating useful ranking ability, but its recall of 0.216 was low at the default classification threshold. It predicted the majority class too frequently and missed most churned customers.

The dummy classifier demonstrated why accuracy is unsuitable as the primary metric. Despite achieving approximately 79.7% accuracy, it failed to identify a single churned customer and had an F1 score of zero.

None of the unbalanced baseline models exceeded an F1 score of 0.59 on the validation set. The random forest came closest at 0.587 and is the strongest initial candidate for imbalance correction and hyperparameter tuning. Logistic regression may also benefit from class weighting or threshold adjustment because of its comparatively strong AUC-ROC.

## 9. Baseline conclusions

This notebook evaluated four models without correcting the class imbalance:

- The dummy classifier predicted only the majority class and produced an F1 score of zero.
- Logistic regression ranked customers reasonably well but identified relatively few churned customers at the default threshold.
- The unrestricted decision tree improved recall and F1 but substantially overfit the training data.
- The random forest provided the strongest validation performance, with an F1 score of 0.587 and an AUC-ROC of 0.855.

These results confirm that the original class imbalance influences the models' ability to identify churned customers. The next stage will test at least two imbalance-handling approaches, including class weighting and resampling, while tuning model hyperparameters using the training and validation sets.

The test set has not been used and will remain untouched until final evaluation.